In [13]:
import urllib.request
import zipfile
import os
from pathlib import Path

url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path= "sms_spam_collection.zip"
extracted_path= "sms_spam_collection"
data_file_path= Path(extracted_path)  / "SMSSpamCollection.tsv"

def download_and_unzip_spam_data(
        url, zip_path, extracted_path, data_file_path):
    if data_file_path.exists():  
        print(f"{data_file_path} already exists. Skipping download"
              "and extraction."
              )
        return

    
    with urllib.request.urlopen(url) as response:
        with open(zip_path, "wb" ) as out_file:
            out_file.write(response.read())

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extracted_path)

    original_file_path= Path(extracted_path)/ "SMSSpamCollection"
    os.rename(original_file_path, data_file_path)
    print(f"file donwloaded and saved as {data_file_path}")

download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)




sms_spam_collection\SMSSpamCollection.tsv already exists. Skipping downloadand extraction.


In [14]:
import pandas as pd
df= pd.read_csv(
    data_file_path, sep= "\t", header= None, names= ["Label", "Text"]
)
df

,Label,Text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [15]:
print(df["Label"].value_counts())

Label
ham     4825
spam     747
Name: count, dtype: int64


In [21]:

print( df[df["Label"]== "spam"]  )

     Label                                               Text
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
5     spam  FreeMsg Hey there darling it's been 3 week's n...
8     spam  WINNER!! As a valued network customer you have...
9     spam  Had your mobile 11 months or more? U R entitle...
11    spam  SIX chances to win CASH! From 100 to 20,000 po...
...    ...                                                ...
5537  spam  Want explicit SEX in 30 secs? Ring 02073162414...
5540  spam  ASKED 3MOBILE IF 0870 CHATLINES INCLU IN FREE ...
5547  spam  Had your contract mobile 11 Mnths? Latest Moto...
5566  spam  REMINDER FROM O2: To get 2.50 pounds free call...
5567  spam  This is the 2nd time we have tried 2 contact u...

[747 rows x 2 columns]


In [16]:
def create_balanced_dataset(df):
    num_spam= df[df["Label"]== "spam"].shape[0]
    ham_subset= df[df["Label"]== "ham"].sample(
        num_spam, random_state=123
    )

    balanced_df = pd.concat([
        ham_subset, df[df["Label"]== "spam"]
    ])
    return balanced_df

balanced_df= create_balanced_dataset(df)
print(balanced_df["Label"].value_counts())

Label
ham     747
spam    747
Name: count, dtype: int64


In [17]:
balanced_df["Label"]= balanced_df["Label"].map({"ham":0, "spam":1})

In [22]:
def random_split(df, train_frac, validation_frac):

    df= df.sample(
        frac=1, random_state= 123
    ).reset_index(drop = True)
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[: train_end]
    validation_df= df[train_end: validation_end]

    test_df= df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df= random_split(
    balanced_df, 0.7, 0.1
)

In [23]:
train_df.to_csv("train.csv", index= None)
validation_df.to_csv("validation_csv", index= None)
test_df.to_csv("test_csv", index= None)

In [30]:
import tiktoken
tokenizer= tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special= {"<|endoftext|>"}))

[50256]


In [31]:
print(tokenizer.decode([50256]))

<|endoftext|>


In [32]:
print(tokenizer.decode([91, 437, 1659, 5239, 91]))

|endoftext|


dataset class

In [ ]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokeniser, max_length=None,
                 pad_tokens_id= 50256):
        self.data= pd.read_csv(csv_file)

        self.encoded_text = [
            tokenizer.encode(text)
        ]